In [1]:
import itertools

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import mesh_utils

Mesh = jax.sharding.Mesh
NamedSharding = jax.sharding.NamedSharding
P = jax.sharding.PartitionSpec

/home/lsiyuan_google_com/miniconda3/envs/torch312/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
mesh = Mesh(mesh_utils.create_device_mesh((8,)), axis_names=('model',))

In [3]:
dtype = jnp.bfloat16
n_tokens = 128
sharding = NamedSharding(mesh, P())
x = jnp.arange(n_tokens).astype(dtype)
x = jax.device_put(x, sharding)

In [4]:
output = jnp.zeros_like(x)

In [5]:
n_chips = len(jax.devices())

def gen_length(n_tokens, n_partitions):
    # low = 0
    # high = 2 * n_tokens // n_partition
    # print(f"low: {low}, high: {high}")
    # len = []
    # remaining_tokens = n_tokens
    # for _ in range(n_partition):
    #     random_int = np.random.randint(low, high)
    #     random_int = min(remaining_tokens, random_int)
    #     remaining_tokens -= random_int
    #     print(random_int)
    bins = [0] * n_partitions
    for _ in range(n_tokens):
        expert_shard_idx = np.random.randint(0, n_partitions)
        assert expert_shard_idx < len(bins)
        bins[expert_shard_idx] += 1
    return bins

num_expert_assigned_tokens = gen_length(n_tokens, n_chips)
print(num_expert_assigned_tokens)
print(sum(num_expert_assigned_tokens))
input_offset = [0] + list(itertools.accumulate(num_expert_assigned_tokens))[:-1]
print(f"input_offset is {input_offset}")

[21, 16, 22, 11, 13, 13, 16, 16]
128
input_offset is [0, 21, 37, 59, 70, 83, 96, 112]
